In [1]:
import ee
import pandas as pd
import numpy as np
import os
import time
# from concurrent.futures import ThreadPoolExecutor, as_completed
# from threading import Lock
from tqdm.auto import tqdm

# Initialize Earth Engine with High Volume Endpoint
try:
    ee.Initialize(project='ee-gsingh')
    # , opt_url='https://earthengine-highvolume.googleapis.com'
except Exception as e:
    print(f"Failed to initialize GEE High Volume: {e}")
    ee.Initialize(project='ee-gsingh')

In [5]:
def retry(max_retries=3, backoff_factor=2):
    """Decorator for retrying GEE calls with exponential backoff."""
    def decorator(func):
        def wrapper(*args, **kwargs):
            retries = 0
            while retries < max_retries:
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    retries += 1
                    if retries == max_retries:
                        raise e
                    sleep_time = backoff_factor ** retries
                    time.sleep(sleep_time)
            return None
        return wrapper
    return decorator

def get_s2_composite(roi, start_date='2022-01-01', end_date='2023-01-01', cs_threshold=0.6):
    """Creates a Sentinel-2 median composite with Cloud Score+ masking and indices."""
    
    # Sentinel-2 Harmonized Collection
    s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterDate(start_date, end_date)
          .filterBounds(roi))
    
    # Cloud Score+ Collection
    cs_plus = (ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
               .filterDate(start_date, end_date)
               .filterBounds(roi))
    
    # Link collections
    linked = s2.linkCollection(cs_plus, ['cs'])
    
    def mask_clouds_and_add_indices(img):
        # Mask clouds using Cloud Score+ 'cs' band
        is_clear = img.select('cs').gte(cs_threshold)
        
        # Scale
        scaled = img.divide(10000).updateMask(is_clear)
        
        # Indices
        nbr = scaled.normalizedDifference(['B8', 'B12']).rename('NBR')
        ndmi = scaled.normalizedDifference(['B8', 'B11']).rename('NDMI')
        ndwi = scaled.normalizedDifference(['B3', 'B8']).rename('NDWI')
        
        return scaled.addBands([nbr, ndmi, ndwi])

    composite = (linked.limit(3).map(mask_clouds_and_add_indices)
                 .select(['NBR', 'NDMI', 'NDWI'])
                 .median())
    
    return composite

@retry()
def get_s2_footprint(point, start_date, end_date):
    """
    Finds the first Sentinel-2 image intersecting the point and returns its footprint.
    """
    col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterBounds(point)
           .filterDate(start_date, end_date))
    
    # Check if we got an image
    size = col.size().getInfo()
    if size == 0:
        return None
    
    return col.first().geometry()

def extract_efficiently(asset_id, output_csv, start_date='2022-01-01', end_date='2023-01-01'):
    """
    Extracts indices using the user-specified logic flow:
    remove processed points -> Iterate remaining points (select a point) -> Find S2 Image -> Get BBox -> Filter Points by BBox -> ReduceRegions -> Pandas Export
    """
    
    # 1. Deduplication: Load processed IDs from existing CSV
    processed_ids = set()
    if os.path.exists(output_csv):
        try:
            if os.path.getsize(output_csv) > 0:
                # Read just the ID column to be fast
                existing_df = pd.read_csv(output_csv, usecols=['id'])
                if 'id' in existing_df.columns:
                    processed_ids = set(existing_df['id'].astype(str))
                print(f"Resuming: {len(processed_ids)} points already processed.")
        except Exception as e:
            print(f"Error reading existing CSV: {e}")

    # 2. filter processed IDs and select first unprocessed point
    # User logic: Filter processed IDs at the start
    def filter_processed_ids(asset_id):
        fscs = ee.FeatureCollection(asset_id)
        if processed_ids:
            print(f"Server-side filtering of {len(processed_ids)} processed IDs...")
            fscs_filtered = fscs.filter(ee.Filter.inList('id', list(processed_ids)).Not())
        else:
            print(f"No processed IDs found, processing all points...")
            fscs_filtered = fscs
        
        return fscs_filtered
    
    # 3. Extract data
    unprocessed = filter_processed_ids(asset_id).limit(5)
    # Check if there are points to process
    if unprocessed.size().getInfo() == 0:
        print("No unprocessed points found.")
        return

    # Use geometry of the first point directly
    # Using limit(1).geometry() returns the geometry of the collection (union of 1 point -> point geometry)
    # This avoids casting issues with ee.Feature()
    search_geom = unprocessed.limit(1).geometry()
    
    bbx = get_s2_footprint(search_geom, start_date, end_date)
    
    if bbx is None:
        print("No Sentinel-2 image found for the first point.")
        return
    points = unprocessed.filterBounds(bbx)
    pts_wid = points.map(lambda ft: ft.set('id', ft.get('system:index')))
    composite = get_s2_composite(bbx, start_date, end_date)
    # 5. reduceRegions
    eedf = composite.reduceRegions(
        collection=pts_wid.limit(5),
        reducer=ee.Reducer.first(),
        scale=10
    )
    eedf
    
    # 6. computeFeatures -> Pandas
    df_result = ee.data.computeFeatures({
        'expression': eedf,
        'fileFormat': 'PANDAS_DATAFRAME'
    })

    if not df_result.empty:
        # 7. Append to CSV
        header = not os.path.exists(output_csv) or os.path.getsize(output_csv) == 0
        
        # Lock if using threads, but we are sequential here as per snippet
        df_result.to_csv(output_csv, mode='a', header=header, index=False)

In [22]:
# import pandas as pd
# from gee_extraction import extract_efficiently
# import os
# import ee
from geeml.utils import eeprint

# Asset ID provided by user
ASSET_ID = "projects/ee-gsingh/assets/RECOVER/fscs_aef_samples"
OUTPUT_CSV = "test_extracted_indices.csv"

def run_test():
    if os.path.exists(OUTPUT_CSV):
        print(f"Removing existing test output: {OUTPUT_CSV}")
        os.remove(OUTPUT_CSV)

    print(f"Starting extraction for asset: {ASSET_ID}", flush=True)
       
    try:
        # Run extraction
        extract_efficiently(ASSET_ID, OUTPUT_CSV)
        
        # Verify Output
        # if os.path.exists(OUTPUT_CSV):
        #     res = pd.read_csv(OUTPUT_CSV)
        #     print("\nExtraction Results Summary:")
        #     print(res.head())
        #     print(f"Total rows: {len(res)}")
        # else:
        #     print("\nWarning: Output CSV not created (might mean no points were processed or all failed).")
            
    except Exception:
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    run_test()


Starting extraction for asset: projects/ee-gsingh/assets/RECOVER/fscs_aef_samples
No processed IDs found, processing all points...


In [23]:
points.first()

In [18]:
import geemap

points = ee.FeatureCollection(ASSET_ID)
start_date = '2022-01-01'
end_date = '2023-01-01'
bbx = get_s2_footprint(points.first().geometry(), start_date, end_date)
filtered = points.filterBounds(bbx)
pts_wid = filtered.map(lambda ft: ft.set('id', ft.get('system:index')))
composite = get_s2_composite(bbx, start_date, end_date)
# 5. reduceRegions
eedf = composite.reduceRegions(
    collection=pts_wid,
    reducer=ee.Reducer.first(),
    scale=10
)
df_result = ee.data.computeFeatures({
    'expression': eedf.limit(5),
    'fileFormat': 'PANDAS_DATAFRAME'
})
print(df_result)
Map = geemap.Map()
Map.addLayer(bbx)
Map.centerObject(points.first(), 12)
Map.addLayer(filtered)
Map

                                                 geo       A00       A01  \
0  {'type': 'Point', 'coordinates': [26.612004268...  0.038447  0.103406   
1  {'type': 'Point', 'coordinates': [26.273072457...  0.004983  0.214133   
2  {'type': 'Point', 'coordinates': [27.077155395... -0.008858  0.284444   
3  {'type': 'Point', 'coordinates': [26.341791663...  0.051734  0.214133   
4  {'type': 'Point', 'coordinates': [26.787358417...  0.044844  0.214133   

        A02       A03       A04       A05       A06       A07       A08  ...  \
0 -0.075356  0.044844 -0.079723 -0.084214  0.032541  0.192910  0.007443  ...   
1 -0.084214  0.022207 -0.119093 -0.035433  0.103406  0.199862  0.113741  ...   
2 -0.041584 -0.041584 -0.160000 -0.013841  0.079723  0.192910  0.066990  ...   
3  0.013841 -0.004983 -0.119093 -0.059116  0.119093  0.199862  0.153787  ...   
4  0.022207 -0.055363 -0.135886 -0.003937  0.108512  0.236463  0.113741  ...   

        A59       A60       A61       A62       A63   NBR  NDM

Map(center=[-30.973058608468772, 26.61200426898607], controls=(WidgetControl(options=['position', 'transparent…

In [15]:
points.limit(3).size()

In [3]:
import ee
from geeml.utils import eeprint
import geemap

try:
    ee.Initialize(project='ee-gsingh')
    # , opt_url='https://earthengine-highvolume.googleapis.com'
except Exception as e:
    print(f"Failed to initialize GEE High Volume: {e}")
    ee.Initialize(project='ee-gsingh')

points = ee.FeatureCollection('projects/ee-gsingh/assets/RECOVER/fscs_aef_samples')
start_date='2022-01-01'
end_date='2023-01-01'

# Map = geemap.Map()
# point = points.filter(ee.Filter.eq('system:index', '0000000000000000cdbd'))
# bbx = get_s2_footprint(point, start_date = start_date, end_date = end_date)
# filtered_pts = points.filterBounds(bbx)
# Map.addLayer(point)
# Map.addLayer(bbx)
# Map.addLayer(filtered_pts)
# Map.centerObject(point, 13)
# Map

In [25]:
points = ee.FeatureCollection('projects/ee-gsingh/assets/RECOVER/fscs_aef_samples')
print(points.size().getInfo())

point = points.sort('system:index').limit(1).geometry()
bbx = get_s2_footprint(point, start_date = start_date, end_date = end_date)
filtered_pts = points.filterBounds(bbx)
print(filtered_pts.size().getInfo())
pts_wid = filtered_pts.map(lambda ft: ft.set('id', ft.get('system:index')))

unprocessed = points.filter(ee.Filter.inList('system:index', pts_wid.aggregate_array('id')).Not())
print(unprocessed.size().getInfo())
# point2 = unprocessed.sort('system:index').limit(1).geometry()
point2 = ee.Feature(unprocessed.sort('system:index').first()).geometry()
bbx2 = get_s2_footprint(point2, start_date = start_date, end_date = end_date)
filtered_pts2 = points.filterBounds(bbx2)
pts_wid2 = filtered_pts2.map(lambda ft: ft.set('id', ft.get('system:index')))

Map = geemap.Map()
Map.addLayer(bbx,{}, 'bbx')
Map.addLayer(bbx2,{}, 'bbx2')
Map.addLayer(point,{}, 'point')
Map.addLayer(point2,{}, 'point2')
Map.centerObject(bbx, 10)
Map


329600
2694
326906


Map(center=[-32.19732936949201, 25.87059343057962], controls=(WidgetControl(options=['position', 'transparent_…